1.Bank Account — Race Condition

In [3]:
import threading
import time

class BankAccount:
    """Rekening bank sederhana. use_lock menentukan apakah operasi dilindungi Lock."""
    def __init__(self, saldo_awal=1_000_000):
        self.saldo = saldo_awal
        self.lock = threading.Lock()

    def _tarik(self):
        temp = self.saldo      # 1) BACA
        time.sleep(0)          # jeda kecil -> beri kesempatan thread lain berjalan
        temp -= 1              # 2) HITUNG
        self.saldo = temp      # 3) TULIS

    def _setor(self):
        temp = self.saldo
        time.sleep(0)
        temp += 1
        self.saldo = temp

    def tarik_setor(self, n_operasi=10_000, use_lock=False):
        for _ in range(n_operasi):
            if use_lock:
                with self.lock:
                    self._tarik()
                with self.lock:
                    self._setor()
            else:
                self._tarik()
                self._setor()


def jalankan_simulasi(use_lock, n_thread=10, n_operasi=10_000, saldo_awal=1_000_000):
    rekening = BankAccount(saldo_awal)
    threads = [
        threading.Thread(target=rekening.tarik_setor, args=(n_operasi, use_lock))
        for _ in range(n_thread)
    ]

    mulai = time.perf_counter()
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    selesai = time.perf_counter()

    return rekening.saldo, selesai - mulai

a.Tanpa Lock

In [4]:
SALDO_AWAL = 1_000_000
N_THREAD = 10
N_OPERASI = 10_000

print(f"Saldo awal        : {SALDO_AWAL:,}")
print(f"Jumlah thread     : {N_THREAD}")
print(f"Operasi per thread: {N_OPERASI:,} (tarik 1, setor 1)")
print("-" * 55)

hasil_tanpa_lock = []
for percobaan in range(10):
    saldo_akhir, durasi = jalankan_simulasi(use_lock=False, n_thread=N_THREAD, n_operasi=N_OPERASI, saldo_awal=SALDO_AWAL)
    selisih = saldo_akhir - SALDO_AWAL
    hasil_tanpa_lock.append(saldo_akhir)
    status = "BENAR" if selisih == 0 else "SALAH (race condition!)"
    print(f"Percobaan {percobaan+1}: saldo akhir = {saldo_akhir:,} (selisih {selisih:+,}) -> {status} | waktu: {durasi:.3f}s")

Saldo awal        : 1,000,000
Jumlah thread     : 10
Operasi per thread: 10,000 (tarik 1, setor 1)
-------------------------------------------------------
Percobaan 1: saldo akhir = 999,999 (selisih -1) -> SALAH (race condition!) | waktu: 0.498s
Percobaan 2: saldo akhir = 999,998 (selisih -2) -> SALAH (race condition!) | waktu: 0.413s
Percobaan 3: saldo akhir = 1,000,000 (selisih +0) -> BENAR | waktu: 0.455s
Percobaan 4: saldo akhir = 999,999 (selisih -1) -> SALAH (race condition!) | waktu: 0.449s
Percobaan 5: saldo akhir = 999,997 (selisih -3) -> SALAH (race condition!) | waktu: 0.596s
Percobaan 6: saldo akhir = 999,999 (selisih -1) -> SALAH (race condition!) | waktu: 0.599s
Percobaan 7: saldo akhir = 999,999 (selisih -1) -> SALAH (race condition!) | waktu: 0.513s
Percobaan 8: saldo akhir = 999,998 (selisih -2) -> SALAH (race condition!) | waktu: 0.532s
Percobaan 9: saldo akhir = 999,998 (selisih -2) -> SALAH (race condition!) | waktu: 0.531s
Percobaan 10: saldo akhir = 999,997 (selis

Tetapi karena beberapa thread membaca dan menulis saldo pada waktu yang hampir bersamaan, hasil dapat menjadi tidak sesuai.

b. Menggunakan Lock

In [6]:
hasil_dengan_lock = []
for percobaan in range(10):
    saldo_akhir, durasi = jalankan_simulasi(use_lock=True, n_thread=N_THREAD, n_operasi=N_OPERASI, saldo_awal=SALDO_AWAL)
    selisih = saldo_akhir - SALDO_AWAL
    hasil_dengan_lock.append(saldo_akhir)
    status = "BENAR" if selisih == 0 else "SALAH (race condition!)"
    print(f"Percobaan {percobaan+1}: saldo akhir = {saldo_akhir:,} (selisih {selisih:+,}) -> {status} | waktu: {durasi:.3f}s")

Percobaan 1: saldo akhir = 1,000,000 (selisih +0) -> BENAR | waktu: 5.021s
Percobaan 2: saldo akhir = 1,000,000 (selisih +0) -> BENAR | waktu: 4.074s
Percobaan 3: saldo akhir = 1,000,000 (selisih +0) -> BENAR | waktu: 4.072s
Percobaan 4: saldo akhir = 1,000,000 (selisih +0) -> BENAR | waktu: 4.820s
Percobaan 5: saldo akhir = 1,000,000 (selisih +0) -> BENAR | waktu: 6.956s
Percobaan 6: saldo akhir = 1,000,000 (selisih +0) -> BENAR | waktu: 8.111s
Percobaan 7: saldo akhir = 1,000,000 (selisih +0) -> BENAR | waktu: 7.002s
Percobaan 8: saldo akhir = 1,000,000 (selisih +0) -> BENAR | waktu: 6.364s
Percobaan 9: saldo akhir = 1,000,000 (selisih +0) -> BENAR | waktu: 6.089s
Percobaan 10: saldo akhir = 1,000,000 (selisih +0) -> BENAR | waktu: 5.311s


Lock digunakan untuk memastikan hanya satu thread yang dapat mengubah saldo pada satu waktu. Dengan demikian, operasi tarik dan setor tidak saling bertabrakan. Tanpa Lock, akses bersamaan terhadap data yang sama dapat menyebabkan race condition. Dengan threading.Lock(), akses terhadap saldo menjadi terkontrol sehingga saldo akhir tetap 1.000.000.

2. Producer-Consumer dengan Banyak Consumer

In [ ]:
import threading
import queue
import time

q = queue.Queue()

def producer():
    for i in range(1, 21):
        q.put(i)
        print(f"Producer menghasilkan: {i}")
        time.sleep(0.05)

    # Sinyal bahwa producer sudah selesai
    q.put(None)


def consumer(nama):
    while True:
        data = q.get()

        if data is None:
            print(f"{nama} berhenti")
            
            # Teruskan sinyal None ke consumer berikutnya
            q.put(None)
            q.task_done()
            break

        print(f"{nama} mengambil data: {data}")
        time.sleep(0.1)

        q.task_done()


producer_thread = threading.Thread(target=producer)

consumer_threads = []

for i in range(3):
    t = threading.Thread(
        target=consumer,
        args=(f"Consumer-{i+1}",)
    )
    consumer_threads.append(t)
    t.start()

producer_thread.start()

producer_thread.join()

q.join()

for t in consumer_threads:
    t.join()

print("Semua thread selesai.")

Producer menghasilkan: 1Consumer-1 mengambil data: 1

Producer menghasilkan: 2
Consumer-2 mengambil data: 2
Producer menghasilkan: 3
Consumer-3 mengambil data: 3
Producer menghasilkan: 4
Consumer-1 mengambil data: 4
Producer menghasilkan: 5
Consumer-2 mengambil data: 5
Producer menghasilkan: 6
Consumer-1 mengambil data: 6
Producer menghasilkan: 7
Consumer-3 mengambil data: 7
Producer menghasilkan: 8
Consumer-2 mengambil data: 8
Producer menghasilkan: 9
Consumer-1 mengambil data: 9
Producer menghasilkan: 10
Consumer-3 mengambil data: 10
Producer menghasilkan: 11Consumer-2 mengambil data: 11

Producer menghasilkan: 12
Consumer-1 mengambil data: 12
Producer menghasilkan: 13
Consumer-3 mengambil data: 13
Producer menghasilkan: 14
Consumer-2 mengambil data: 14
Producer menghasilkan: 15Consumer-1 mengambil data: 15

Producer menghasilkan: 16
Consumer-3 mengambil data: 16
Producer menghasilkan: 17
Consumer-2 mengambil data: 17
Producer menghasilkan: 18
Consumer-1 mengambil data: 18
Producer m

Penggunaan Queue memudahkan pertukaran data antara producer dan consumer secara aman. Dengan multithreading, ketiga consumer dapat bekerja secara bersamaan sehingga proses menjadi lebih efisien.

3. Eksperimen ThreadPoolExecutor

In [1]:
import time
import random
from concurrent.futures import ThreadPoolExecutor


# Fungsi untuk simulasi pemrosesan item
def proses_item(id_item):
    waktu_proses = random.uniform(0.05, 0.15)

    # Simulasi proses I/O
    time.sleep(waktu_proses)

    return f"Item-{id_item} selesai"


# Fungsi untuk melakukan pengujian
def jalankan_pengujian(jumlah_worker, jumlah_item=50):

    daftar_item = list(range(1, jumlah_item + 1))

    waktu_awal = time.perf_counter()

    # ThreadPoolExecutor
    with ThreadPoolExecutor(max_workers=jumlah_worker) as executor:
        hasil = list(
            executor.map(proses_item, daftar_item)
        )

    waktu_akhir = time.perf_counter()

    total_waktu = waktu_akhir - waktu_awal

    return total_waktu


# Program utama
if __name__ == "__main__":

    JUMLAH_ITEM = 50

    # Sesuai soal nomor 3
    DAFTAR_WORKER = [2, 4, 8, 16]

    print("=" * 65)
    print("       EKSPERIMEN THREADPOOL EXECUTOR")
    print("=" * 65)

    print(
        f"{'Worker':<12}"
        f"{'Jumlah Item':<15}"
        f"{'Waktu (detik)':<20}"
        f"{'Speedup':<12}"
    )

    print("-" * 65)

    waktu_awal = None

    for worker in DAFTAR_WORKER:

        waktu = jalankan_pengujian(
            worker,
            JUMLAH_ITEM
        )

        # Worker = 2 digunakan sebagai baseline
        if waktu_awal is None:
            waktu_awal = waktu
            speedup = "1.00x"
        else:
            speedup = f"{waktu_awal / waktu:.2f}x"

        print(
            f"{worker:<12}"
            f"{JUMLAH_ITEM:<15}"
            f"{waktu:<20.4f}"
            f"{speedup:<12}"
        )

    print("=" * 65)

       EKSPERIMEN THREADPOOL EXECUTOR
Worker      Jumlah Item    Waktu (detik)       Speedup     
-----------------------------------------------------------------
2           50             2.5643              1.00x       
4           50             1.2575              2.04x       
8           50             0.6929              3.70x       
16          50             0.3880              6.61x       


Menambah max_workers tidak selalu mempercepat proses. Pada jumlah tertentu terjadi titik jenuh, ketika penambahan thread tidak lagi memberikan peningkatan kecepatan karena adanya keterbatasan CPU, memori, I/O, dan overhead pengelolaan thread. Jadi, jumlah worker yang terlalu banyak justru bisa membuat proses tetap sama atau lebih lambat.

4. Refleksi Singkat,
Threading Python dapat membantu ketika program melakukan pekerjaan yang banyak menunggu I/O, misalnya mengunduh beberapa file dari internet, membaca banyak file, atau mengakses beberapa layanan jaringan secara bersamaan. Dengan threading, ketika satu thread sedang menunggu proses I/O selesai, thread lain dapat mengerjakan tugas lainnya sehingga waktu tunggu dapat dimanfaatkan. Sebaliknya, threading tidak selalu membantu untuk pekerjaan yang membutuhkan komputasi CPU secara berat, seperti pengolahan gambar dalam jumlah besar, perhitungan matematika kompleks, atau pemrosesan data yang membutuhkan CPU secara terus-menerus. Pada kondisi tersebut, penggunaan threading Python dibatasi oleh GIL sehingga beberapa thread tidak dapat menjalankan bytecode Python secara paralel pada satu waktu. Untuk pekerjaan CPU-bound, multiprocessing lebih sesuai karena proses yang berbeda dapat menggunakan core CPU secara paralel.